# CLUSTER HUNTER — find the next GEHC\nThe exact method that found the Culp cluster, as one machine. Three cells, run top to bottom (~10-15 min):\n1. **Candidates** — every insider cluster-buy in the market (two feeds, one pass)\n2. **Quality grade** — the GEHC tests on real filings: 3+ distinct buyers (or 2 + C-suite), $500K+, ZERO discretionary sells, anti-theater check (uniform same-day buys auto-fail)\n3. **Price + health gates** — 20%+ off highs, still within 20% of the insiders' own average price, profitable, FCF-positive, revenue not shrinking\n\n**Output = candidates for name-level work, NOT buys.** GEHC itself needed the China/imaging question answered before sizing. Cadence: weekly, and always in the 2 weeks after each earnings season (open trading windows = when clusters print). Nothing runs unattended.

In [ ]:
# 1 · CANDIDATES — market-wide insider cluster feeds (two OpenInsider pages, one pass)
%pip -q install yfinance
import pandas as pd, requests
from io import StringIO

UA = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
PAGES = ["http://openinsider.com/latest-cluster-buys",
         "http://openinsider.com/top-officer-purchases-of-the-month"]
ticks = set()
for u in PAGES:
    try:
        tabs = pd.read_html(StringIO(requests.get(u, headers=UA, timeout=30).text))
        tab = next(x for x in tabs if x.shape[1] > 8)
        tab.columns = [str(c).replace("\xa0", " ").strip() for c in tab.columns]
        tcol = next(c for c in tab.columns if "Ticker" in c)
        for t in tab[tcol].astype(str):
            t = t.strip().upper()
            if t and t.isalpha() and len(t) <= 5:
                ticks.add(t)
        print(f"{u.split('/')[-1]}: ok")
    except Exception as e:
        print(f"{u.split('/')[-1]}: {str(e)[:40]}")
CANDS = sorted(ticks)
print(f"\n{len(CANDS)} candidate tickers:", " ".join(CANDS))


In [ ]:
# 2 · CLUSTER QUALITY GRADE — EDGAR Form 4 parse, the GEHC tests
# A-grade: >=3 distinct discretionary buyers, >=$500K total, ZERO discretionary sells
# B-grade: >=2 buyers incl C-suite (CEO/CFO/Pres/GC/Chair), >=$300K, zero disc sells
# anti-theater: uniform sizes on one day (EPAM pattern) = auto-fail
import requests, time, re
import pandas as pd
import xml.etree.ElementTree as ET
from datetime import date, timedelta

H = {"User-Agent": "Jake research 7bm4q6x5sm@privaterelay.appleid.com"}
DAYS = 120
SINCE = (date.today() - timedelta(days=DAYS)).isoformat()
CSUITE = ("CEO","CFO","PRES","GC","CHAIR","CHIEF","COUNSEL")

cm = requests.get("https://www.sec.gov/files/company_tickers.json", headers=H, timeout=30).json()
cikmap = {v["ticker"]: str(v["cik_str"]).zfill(10) for v in cm.values()}

def txt(el, path):
    e = el.find(path)
    return e.text.strip() if e is not None and e.text else None

graded = {}
for t in CANDS:
    cik = cikmap.get(t)
    if not cik: continue
    try:
        sub = requests.get(f"https://data.sec.gov/submissions/CIK{cik}.json", headers=H, timeout=30).json()
    except Exception: continue
    time.sleep(0.12)
    rec = sub["filings"]["recent"]
    f4s = [(rec["accessionNumber"][i], rec["filingDate"][i], rec["primaryDocument"][i])
           for i in range(len(rec["form"])) if rec["form"][i] == "4" and rec["filingDate"][i] >= SINCE]
    buys, sells = [], 0
    for acc, fdate, pdoc in f4s[:30]:
        url = f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{acc.replace('-','')}/{pdoc.split('/')[-1]}"
        try:
            x = requests.get(url, headers=H, timeout=30).text
            time.sleep(0.12)
            root = ET.fromstring(re.sub(r"<\?xml[^>]*\?>", "", x, count=1))
        except Exception: continue
        owner = txt(root, ".//reportingOwner/reportingOwnerId/rptOwnerName") or "?"
        rel = root.find(".//reportingOwner/reportingOwnerRelationship")
        title = (txt(rel, "officerTitle") or ("Director" if txt(rel, "isDirector") in ("1","true") else "?"))
        plan = txt(root, ".//aff10b5One") in ("1","true")
        for tr in root.findall(".//nonDerivativeTransaction"):
            c_ = txt(tr, "transactionCoding/transactionCode")
            sh = txt(tr, "transactionAmounts/transactionShares/value")
            pr = txt(tr, "transactionAmounts/transactionPricePerShare/value")
            dt_ = txt(tr, "transactionDate/value")
            if not (c_ and sh): continue
            v = float(sh) * float(pr or 0)
            if c_ == "P":
                buys.append({"owner": owner, "title": title.upper(), "v": v, "px": float(pr or 0), "d": dt_})
            elif c_ == "S" and not plan:
                sells += v
    if not buys: continue
    b = pd.DataFrame(buys)
    tot, n = b.v.sum(), b.owner.nunique()
    has_cs = b.title.str.contains("|".join(CSUITE)).any()
    per = b.groupby("owner").v.sum()
    theater = (n >= 3 and per.max()/max(per.min(),1) < 1.3 and b.d.nunique() == 1)
    grade = None
    if sells == 0 and not theater:
        if n >= 3 and tot >= 500000: grade = "A"
        elif n >= 2 and has_cs and tot >= 300000: grade = "B"
    if grade:
        wavg = (b.v.sum() / (b.v / b.px).sum()) if (b.px > 0).all() else None
        graded[t] = {"grade": grade, "buyers": n, "total_$": int(tot), "c_suite": has_cs,
                     "ins_avg_px": round(wavg, 2) if wavg else None,
                     "last_buy": b.d.max(), "disc_sells_$": int(sells)}
    print(".", end="")

gq = pd.DataFrame(graded).T
print(f"\n\n{len(gq)} clusters pass the GEHC quality tests:")
print(gq.to_string() if len(gq) else "none")


In [ ]:
# 3 · PRICE + HEALTH GATES — on sale, near the insiders' price, business intact
# Gates: dd <= -20% vs 2y high | spot within +/-20% of insider avg px | profitable | FCF+ | revenue not shrinking
import yfinance as yf, numpy as np, pandas as pd, time

final = []
for t, g in graded.items():
    try:
        tk = yf.Ticker(t)
        h = tk.history(period="2y")["Close"]
        if len(h) < 200: continue
        spot = h.iloc[-1]
        dd = (spot/h.max() - 1) * 100
        if dd > -20: continue
        near = abs(spot / g["ins_avg_px"] - 1) * 100 if g["ins_avg_px"] else None
        if near is not None and near > 20: continue           # insiders' zone left behind
        i = tk.info
        pe, fcf, mc = i.get("trailingPE"), i.get("freeCashflow"), i.get("marketCap")
        rg = i.get("revenueGrowth")
        prof = pe is not None and pe > 0
        fcfy = round(fcf/mc*100, 1) if (fcf and mc) else None
        if not prof: continue
        if fcfy is not None and fcfy < 0: continue
        if rg is not None and rg < -0.05: continue
        final.append({"ticker": t, "grade": g["grade"], "buyers": g["buyers"],
                      "cluster_$": g["total_$"], "c_suite": g["c_suite"],
                      "spot": round(spot,2), "ins_avg": g["ins_avg_px"],
                      "vs_ins%": round((spot/g["ins_avg_px"]-1)*100,1) if g["ins_avg_px"] else None,
                      "dd%": round(dd,0), "PE": round(pe,1) if pe else None,
                      "FCFy%": fcfy, "revG%": round(rg*100,1) if rg is not None else None,
                      "mcap_$B": round(mc/1e9,1) if mc else None, "last_buy": g["last_buy"]})
    except Exception: pass
    time.sleep(0.4)

out = pd.DataFrame(final)
print("=== GEHC CLONES — quality cluster + on sale + insider zone + intact business ===")
print(out.sort_values("cluster_$", ascending=False).to_string(index=False) if len(out)
      else "NONE this run — the bar is the point. Re-run weekly and after earnings seasons.")
print("\nNext step per name: E-path work (why is it down, is the E broken) BEFORE any dollar.")


*Quality hierarchy reminder: varied sizes beat uniform (anti-EPAM) · C-suite+GC beats directors-only · buying INTO strength (rising prices across the cluster) is the rare strong variant · the architect/founder buying is the loudest single print · insiders flipping to sells = the exit signal on any name this finds.*